# 🚀 Unidad 3 — Clase 3: Shell Sort y Counting Sort

## Información del Curso

| Aspecto | Detalle |
|--------|--------|
| **Universidad** | Universidad de Talca, Chile |
| **Carrera** | Ingeniería Civil en Informática |
| **Semestre** | 2°-3° año |
| **Curso** | Algoritmos y Estructuras de Datos |
| **Docente** | PhD. César Astudillo |
| **Clase** | Unidad 3, Clase 3 — Shell Sort + Counting Sort |
| **Duración** | 50 minutos |

---
> 🎯 *Este notebook está diseñado para ser ejecutado en clase de forma interactiva.  
> Ejecuta las celdas en orden de arriba hacia abajo.*

## Verificación de Dependencias

In [ ]:
import sys
required = {'numpy': 'numpy', 'matplotlib': 'matplotlib', 'ipywidgets': 'ipywidgets'}
for nombre, paquete in required.items():
    try:
        __import__(paquete)
        print(f"✅ {nombre} instalado correctamente")
    except ImportError:
        print(f"❌ {nombre} NO encontrado — instala con: pip install {paquete}")
print("\n🐍 Python", sys.version.split()[0], "| Todo listo para comenzar.")

## 🎯 Objetivos de Aprendizaje

Al finalizar esta sesión, el estudiante será capaz de:

1. **Explicar** por qué Insertion Sort falla con datos lejanos de su posición final.
2. **Implementar** Shell Sort con distintas secuencias de gaps y comparar su rendimiento.
3. **Comprender** por qué la complejidad de Shell Sort depende de la secuencia de gaps elegida.
4. **Demostrar** que existe una cota inferior Ω(n log n) para algoritmos basados en comparaciones.
5. **Implementar** Counting Sort y explicar cuándo rompe la cota inferior.

# Sección 1: La Debilidad de Insertion Sort (5 minutos)

## El problema de los elementos "lejanos"

Considera esta lista: `[9, 8, 7, 6, 5, 4, 3, 2, 1, 0]`

Con Insertion Sort, el elemento `0` empieza en la posición 9 y debe llegar a la posición 0.  
Necesita **9 desplazamientos** de un solo paso cada vez.

```
Paso 9: clave=0, debe recorrer toda la lista hacia la izquierda
        [9, 8, 7, 6, 5, 4, 3, 2, 1, 0]
         ←←←←←←←←←←←←←←←←←←←←←← (9 desplazamientos de a 1)
```

> 📌 **El problema fundamental:** Insertion Sort solo mueve elementos **de a un paso**.  
> Para llevar un elemento de la posición n-1 a la posición 0, necesita n-1 pasos.

> 🎙️ **[PAUSA PROFESOR]** *"¿Cómo podrían modificar Insertion Sort para que mueva elementos más lejos en cada paso?"*

## La solución de Shell (1959)

Donald Shell propuso una idea brillante en su paper de 1959:  
**¿Por qué no hacer primero Insertion Sort con saltos grandes?**

Si primero ordenamos elementos que están **gap** posiciones de distancia,  
los elementos quedan "casi en su lugar" y el Insertion Sort final es rápido.

# Sección 2: Shell Sort (30 minutos)

In [ ]:
# Shell Sort — implementación con secuencia de gaps configurable
def shell_sort(lista: list, gaps: list = None, verbose: bool = False) -> tuple:
    """
    Ordena 'lista' usando Shell Sort con la secuencia de gaps indicada.

    Shell Sort es una generalización de Insertion Sort que primero ordena
    elementos separados por un gap grande, luego reduce el gap progresivamente
    hasta gap=1 (que es Insertion Sort estándar).

    Parámetros:
        lista  (list): lista de elementos comparables
        gaps   (list): secuencia de gaps en orden DECRECIENTE.
                       Si None, usa la secuencia de Knuth: [1, 4, 13, 40, ...]
        verbose (bool): muestra el estado tras cada fase de gap

    Retorna:
        tuple: (lista_ordenada, n_comparaciones, n_swaps)

    Complejidad:
        Temporal: depende de la secuencia de gaps
                  O(n^1.5) con gaps de Knuth (empírico)
                  O(n²)    con gaps de Shell (teórico)
                  O(n log²n) con gaps de Hibbard (teórico)
        Espacial: O(1) extra — in-place
    """
    a = lista[:]
    n = len(a)

    # Si no se especifican gaps, usar secuencia de Knuth: 1, 4, 13, 40, 121, ...
    if gaps is None:
        gaps = []
        g = 1
        while g < n // 3:
            g = 3 * g + 1   # fórmula de Knuth: h = 3h + 1
            gaps.append(g)
        gaps.reverse()
        if not gaps:
            gaps = [1]

    comparaciones = 0
    swaps         = 0

    for gap in gaps:
        # Insertion Sort con paso 'gap' en vez de 1
        for i in range(gap, n):
            clave = a[i]
            j = i - gap
            while j >= 0 and a[j] > clave:
                a[j + gap] = a[j]
                j -= gap
                comparaciones += 1
                swaps += 1
            if j >= 0:
                comparaciones += 1
            a[j + gap] = clave

        if verbose:
            print(f"  Gap={gap:3d}: {a}")

    return a, comparaciones, swaps


# ─── Demo comparativo ─────────────────────────────────────────────────────
import random
random.seed(42)
datos = list(range(10, 0, -1))   # peor caso para Insertion Sort

print("=== Shell Sort — trazado con gaps de Knuth ===")
print(f"Entrada: {datos}")
res, cmp, sw = shell_sort(datos, verbose=True)
print(f"\nResultado: {res}")
print(f"Comparaciones: {cmp} | Swaps: {sw}")

In [ ]:
# Comparación de secuencias de gaps clásicas
def generar_gaps(nombre: str, n: int) -> list:
    """Genera la secuencia de gaps para Shell Sort según el método indicado."""
    if nombre == 'shell':
        # Shell original (1959): n/2, n/4, n/8, ..., 1
        gaps = []
        g = n // 2
        while g >= 1:
            gaps.append(g)
            g //= 2
        return gaps

    elif nombre == 'hibbard':
        # Hibbard (1963): 2^k - 1 → 1, 3, 7, 15, 31, ...  → O(n^1.5) teórico
        gaps = []
        k = 1
        while (2**k - 1) < n:
            gaps.append(2**k - 1)
            k += 1
        return sorted(gaps, reverse=True)

    elif nombre == 'knuth':
        # Knuth (1973): (3^k - 1) / 2 → 1, 4, 13, 40, 121, ...  → O(n^1.5) empírico
        gaps = []
        g = 1
        while g < n // 3:
            g = 3 * g + 1
            gaps.append(g)
        return sorted(gaps, reverse=True) or [1]

    elif nombre == 'ciura':
        # Ciura (2001): secuencia empíricamente óptima conocida
        base = [701, 301, 132, 57, 23, 10, 4, 1]
        return [g for g in base if g < n]

    return [1]

# Mostrar las secuencias para n=100
n = 100
print(f"Secuencias de gaps para n={n}:")
for nombre in ['shell', 'hibbard', 'knuth', 'ciura']:
    gaps = generar_gaps(nombre, n)
    print(f"  {nombre:8s}: {gaps}")

In [ ]:
# Benchmark: Shell Sort con distintas secuencias vs Insertion Sort
import timeit, random

def insertion_sort_simple(a):
    a = a[:]
    for i in range(1, len(a)):
        clave = a[i]; j = i - 1
        while j >= 0 and a[j] > clave:
            a[j+1] = a[j]; j -= 1
        a[j+1] = clave
    return a

ns = [500, 1000, 2000, 5000]
print(f"{'n':>5} | {'Insertion':>10} | {'Shell':>10} | {'Hibbard':>10} | {'Knuth':>10} | {'Ciura':>10}")
print("-" * 65)

for n in ns:
    datos = random.sample(range(n*3), n)
    reps = 20

    t_ins  = timeit.timeit(lambda: insertion_sort_simple(datos), number=reps) / reps
    t_sh   = timeit.timeit(lambda: shell_sort(dados := datos[:], generar_gaps('shell',   n)), number=reps) / reps
    t_hib  = timeit.timeit(lambda: shell_sort(dados := datos[:], generar_gaps('hibbard', n)), number=reps) / reps
    t_knu  = timeit.timeit(lambda: shell_sort(dados := datos[:], generar_gaps('knuth',   n)), number=reps) / reps
    t_ciu  = timeit.timeit(lambda: shell_sort(dados := datos[:], generar_gaps('ciura',   n)), number=reps) / reps

    print(f"{n:>5} | {t_ins*1000:>8.2f}ms | {t_sh*1000:>8.2f}ms | "
          f"{t_hib*1000:>8.2f}ms | {t_knu*1000:>8.2f}ms | {t_ciu*1000:>8.2f}ms")

print("\n→ Todas las variantes de Shell Sort superan a Insertion Sort para n grande.")

## Visualización Animada — Shell Sort

In [ ]:
# Animación de Shell Sort mostrando las fases de gap
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display

try:
    import google.colab; EN_COLAB = True
except ImportError:
    EN_COLAB = False

if not EN_COLAB:
    try:
        get_ipython().run_line_magic('matplotlib', 'widget')
    except Exception:
        get_ipython().run_line_magic('matplotlib', 'inline')

def animar_shell_sort(datos_orig, interval=400):
    """Anima Shell Sort mostrando cada fase de gap con color distinto."""
    a = datos_orig[:]
    n = len(a)
    frames = []

    n_copy = a[:]
    gaps = generar_gaps('knuth', n) or [1]

    # Capturar frames
    frames.append((a[:], -1, -1, gaps[0] if gaps else 1, 'inicio'))
    for gap in gaps:
        for i in range(gap, n):
            clave = a[i]; j = i - gap
            while j >= 0 and a[j] > clave:
                a[j + gap] = a[j]; j -= gap
                frames.append((a[:], j + gap, i, gap, f'gap={gap}'))
            a[j + gap] = clave
            frames.append((a[:], j + gap, i, gap, f'gap={gap}'))
    frames.append((a[:], -1, -1, 1, 'terminado'))

    # Paleta de colores por gap
    import matplotlib.cm as cm
    colores_gap = {}
    unique_gaps = list(dict.fromkeys(f[3] for f in frames))
    cmap = cm.get_cmap('Set2', len(unique_gaps))
    for idx, g in enumerate(unique_gaps):
        colores_gap[g] = cmap(idx)

    COLOR_BASE   = '#90CAF9'
    COLOR_SORTED = '#A5D6A7'

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.set_xlim(-0.5, n - 0.5)
    ax.set_ylim(0, max(datos_orig) + 3)
    ax.set_xlabel('Índice'); ax.set_ylabel('Valor')
    ax.grid(axis='y', alpha=0.3)

    bars   = ax.bar(range(n), a, color=COLOR_BASE, edgecolor='white', linewidth=1.2)
    textos = [ax.text(i, a[i] + 0.3, str(a[i]), ha='center', va='bottom', fontsize=8)
              for i in range(n)]
    info_txt = ax.text(0.02, 0.95, '', transform=ax.transAxes, fontsize=10,
                       verticalalignment='top',
                       bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.7))

    def actualizar(fidx):
        estado, pos1, pos2, gap, label = frames[fidx]
        color_actual = colores_gap.get(gap, COLOR_BASE)
        for i, (bar, txt) in enumerate(zip(bars, textos)):
            bar.set_height(estado[i])
            txt.set_position((i, estado[i] + 0.3))
            txt.set_text(str(estado[i]))
            if label == 'terminado':
                bar.set_color(COLOR_SORTED)
            elif i == pos1 or i == pos2:
                bar.set_color('#FF7043')
            else:
                bar.set_color(color_actual)
        ax.set_title(f'Shell Sort (Knuth gaps) — Fase: {label}', fontsize=12, fontweight='bold')
        info_txt.set_text(f'Frame {fidx}/{len(frames)-1}  |  {label}')
        return bars

    anim = animation.FuncAnimation(fig, actualizar, frames=len(frames),
                                   interval=interval, blit=False, repeat=False)
    plt.tight_layout()
    display(HTML(anim.to_jshtml()))
    plt.close()

import random; random.seed(7)
datos_demo = random.sample(range(1, 22), 12)
print(f"Datos: {datos_demo}")
animar_shell_sort(datos_demo, interval=350)

## Análisis de Complejidad — Shell Sort

La complejidad de Shell Sort es uno de los problemas abiertos más longevos en algoritmia.  
Depende críticamente de la secuencia de gaps elegida:

| Secuencia de gaps | Complejidad teórica | Estado |
|-------------------|--------------------|----|
| Shell: n/2, n/4, ... | O(n²) | Demostrado |
| Hibbard: 2ᵏ-1 | O(n^1.5) | Demostrado |
| Knuth: (3ᵏ-1)/2 | O(n^1.5) empírico | No demostrado formalmente |
| Ciura: 1,4,10,23,... | Desconocida | Mejor empíricamente conocida |
| Sedgewick: mezcla de 4ᵏ+3·2ᵏ+1 | O(n^{4/3}) | Demostrado |

> ⚠️ **Hecho notable:** Para la secuencia de Knuth (la más usada), la complejidad exacta  
> **no está demostrada formalmente** — es un problema abierto desde 1973.

> 💡 **¿Por qué usar Shell Sort si no conocemos su complejidad?**  
> En la práctica, para n ≤ 10.000 es más rápido que O(n log n) como Mergesort porque  
> tiene overhead muy bajo (no necesita memoria extra, sin recursión).

# Sección 3: Rompiendo la Barrera — Counting Sort (15 minutos)

## La Cota Inferior Ω(n log n)

Existe un teorema fundamental que dice:

> 📌 **Teorema:** Todo algoritmo de ordenamiento basado en **comparaciones** necesita  
> al menos $\Omega(n \log n)$ comparaciones en el peor caso.

**Idea de la demostración (árbol de decisión):**

Un algoritmo de comparación puede modelarse como un árbol binario:
- Cada nodo interno: una comparación `a[i] < a[j]` (sí/no)
- Cada hoja: una permutación válida de la salida
- El árbol debe tener **al menos n! hojas** (una por cada permutación posible)
- Un árbol binario con n! hojas tiene altura ≥ log₂(n!) ≥ n log₂(n) - n/ln(2) = Ω(n log n)

$$h \geq \log_2(n!) \approx n\log_2 n - n\log_2 e = \Omega(n \log n)$$

> 🎙️ **[PAUSA PROFESOR]** *"¿Cómo podría un algoritmo NO basarse en comparaciones?"*

## La respuesta: Counting Sort

Counting Sort **no compara elementos entre sí**. En cambio, **cuenta** cuántas veces  
aparece cada valor y usa esa información para colocarlos directamente.

**Restricción crítica:** solo funciona si los valores son **enteros en un rango [0, k) conocido**.

In [ ]:
# Counting Sort — implementación completa y comentada
def counting_sort(lista: list, k: int = None) -> list:
    """
    Ordena una lista de enteros no negativos usando Counting Sort.
    
    Idea: si sabemos que los valores están en [0, k), podemos contar
    cuántas veces aparece cada valor y reconstruir la lista ordenada
    sin necesidad de comparar elementos entre sí.

    Parámetros:
        lista (list): lista de enteros no negativos
        k     (int):  valor máximo + 1 (rango = [0, k))
                      Si None, se calcula automáticamente como max(lista)+1

    Retorna:
        list: nueva lista ordenada

    Complejidad:
        Temporal: O(n + k)  — n para contar, k para reconstruir
        Espacial: O(n + k)  — arreglo de conteos de tamaño k + salida de tamaño n

    Cuándo usar:
        - Los valores son enteros en un rango pequeño conocido
        - k = O(n)  →  complejidad efectiva O(n)
        - k >> n    →  poco eficiente (mejor usar comparación)
    """
    if not lista:
        return []

    if k is None:
        k = max(lista) + 1

    # Paso 1: Contar frecuencias
    conteo = [0] * k
    for valor in lista:
        conteo[valor] += 1   # O(n)

    # Paso 2: Reconstruir la lista ordenada
    resultado = []
    for valor in range(k):          # O(k)
        resultado.extend([valor] * conteo[valor])

    return resultado


# ─── Demo ─────────────────────────────────────────────────────────────────
print("=== Counting Sort — demostración ===")
datos = [4, 2, 2, 8, 3, 3, 1, 0, 4, 2]
print(f"Entrada: {datos}")
print(f"Rango:   [0, {max(datos)}]")
resultado = counting_sort(datos)
print(f"Salida:  {resultado}")

print("\n=== Visualización del arreglo de conteos ===")
k = max(datos) + 1
conteo = [0] * k
for v in datos: conteo[v] += 1
for i, c in enumerate(conteo):
    if c > 0:
        print(f"  conteo[{i}] = {c}  →  {'█' * c} ({c} veces)")

In [ ]:
# Visualización: Counting Sort vs Insertion Sort en función de n y k
import timeit, random, matplotlib.pyplot as plt
import numpy as np

# Escenario 1: k pequeño (k = 10, rango [0, 9])
ns = [100, 500, 1000, 5000, 10000]
k_small = 10

tiempos_cs  = []
tiempos_ins = []

def insertion_sort_simple(a):
    a = a[:]
    for i in range(1, len(a)):
        c = a[i]; j = i - 1
        while j >= 0 and a[j] > c: a[j+1] = a[j]; j -= 1
        a[j+1] = c
    return a

for n in ns:
    datos = [random.randint(0, k_small-1) for _ in range(n)]
    t_cs  = timeit.timeit(lambda: counting_sort(datos, k_small), number=30) / 30
    t_ins = timeit.timeit(lambda: insertion_sort_simple(datos),  number=30) / 30
    tiempos_cs.append(t_cs * 1000)
    tiempos_ins.append(t_ins * 1000)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(ns, tiempos_cs,  'o-', color='#4CAF50', linewidth=2, label=f'Counting Sort (k={k_small})')
ax.plot(ns, tiempos_ins, 's--', color='#2196F3', linewidth=2, label='Insertion Sort')
ax.set_xlabel('Tamaño n'); ax.set_ylabel('Tiempo (ms)')
ax.set_title(f'Counting Sort vs Insertion Sort — rango pequeño k={k_small}')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n{'n':>6} | {'Counting Sort':>14} | {'Insertion Sort':>14}")
print("-" * 40)
for n, tc, ti in zip(ns, tiempos_cs, tiempos_ins):
    print(f"{n:>6} | {tc:>12.3f}ms | {ti:>12.3f}ms")

# Sección 4: Panorama — ¿Cuándo usar qué? (5 minutos)

Con esto cerramos el módulo de **ordenamiento elemental**. Tenemos ahora 4 algoritmos:

| Algoritmo | Complejidad | Memoria extra | Condición de uso |
|-----------|-------------|---------------|-----------------|
| Selection Sort | O(n²) siempre | O(1) | Minimizar swaps (datos en escritura lenta) |
| Insertion Sort | O(n) – O(n²) | O(1) | n pequeño (≤ 30), datos casi ordenados |
| Bubble Sort | O(n) – O(n²) | O(1) | Detección rápida de lista ya ordenada |
| Shell Sort | O(n^1.5) aprox | O(1) | n mediano (100–50.000), sin recursión |
| Counting Sort | O(n + k) | O(k) | Enteros en rango [0, k) pequeño y conocido |

> 💡 **¿Qué viene a continuación?**  
> En la próxima unidad veremos **Merge Sort** y **Quick Sort** — algoritmos O(n log n)  
> que dominan en la práctica. Merge Sort usa Divide & Conquer. Quick Sort usa particiones.  
> La cota inferior Ω(n log n) que vimos hoy nos dice que no podemos hacerlo mejor  
> con comparaciones.

# Sección 5: Widget Interactivo — Explorador de Shell Sort

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import random, timeit

slider_n = widgets.IntSlider(value=20, min=5, max=200, step=5,
                              description='n:', style={'description_width': '120px'})
gap_select = widgets.Dropdown(
    options=[('Knuth (1,4,13,40,...)', 'knuth'),
             ('Hibbard (1,3,7,15,...)', 'hibbard'),
             ('Shell original (n/2,...)', 'shell'),
             ('Ciura (empírico)', 'ciura')],
    value='knuth', description='Gaps:', style={'description_width': '120px'})
tipo_datos = widgets.Dropdown(
    options=[('Aleatorio', 'random'), ('Casi ordenado', 'nearly'), ('Invertido', 'reversed')],
    value='random', description='Datos:', style={'description_width': '120px'})
boton  = widgets.Button(description='▶ Ejecutar', button_style='primary')
salida = widgets.Output()

def al_ejecutar(b):
    with salida:
        salida.clear_output(wait=True)
        n = slider_n.value
        tipo = tipo_datos.value
        nombre_gap = gap_select.value

        if tipo == 'random':   datos = random.sample(range(n*3), n)
        elif tipo == 'nearly':
            datos = list(range(n))
            for _ in range(max(1, n//10)):
                i, j = random.randint(0,n-2), random.randint(0,n-2)
                datos[i], datos[j] = datos[j], datos[i]
        else: datos = list(range(n, 0, -1))

        gaps = generar_gaps(nombre_gap, n) or [1]
        _, cmp, sw = shell_sort(datos, gaps)
        t = timeit.timeit(lambda: shell_sort(datos[:], gaps[:]), number=200) / 200

        print(f"n={n} | tipo={tipo} | gaps={nombre_gap}")
        print(f"Secuencia de gaps: {gaps}")
        print(f"Comparaciones: {cmp} | Swaps: {sw}")
        print(f"Tiempo: {t*1000:.4f} ms")
        print(f"\nEsperado teórico O(n^1.5) ≈ {n**1.5:.0f} operaciones")

boton.on_click(al_ejecutar)
display(widgets.VBox([
    widgets.HBox([slider_n, gap_select, tipo_datos]),
    boton, salida
]))

# Sección 6: Ejercicio Práctico

## 🧪 Ejercicio 1 ⭐: Shell Sort con Gap Personalizado

**Descripción:** Implementa `shell_sort_gaps(lista, gaps)` que ordena usando  
exactamente la secuencia de gaps recibida (sin generar una automáticamente).

**Entrada:** `lista` (lista de enteros), `gaps` (lista de enteros en orden decreciente)  
**Salida:** Nueva lista ordenada

**Ejemplo:**
```
Entrada: lista=[8, 3, 1, 9, 2, 7], gaps=[3, 1]
Salida:  [1, 2, 3, 7, 8, 9]
```

**Restricciones:** El último valor de `gaps` debe ser 1.  
**Complejidad esperada:** O(n²) garantizado con gaps=[1], mejor con gaps más inteligentes

In [ ]:
def shell_sort_gaps(lista: list, gaps: list) -> list:
    """
    Ordena 'lista' usando Shell Sort con la secuencia de gaps dada.

    Parámetros:
        lista (list): lista de enteros
        gaps  (list): secuencia de gaps en orden DECRECIENTE, debe terminar en 1
    Retorna:
        list: nueva lista ordenada
    """
    # Tu código aquí
    pass

In [ ]:
def verificar_ejercicio_1(fn):
    import time
    casos = [
        ([8, 3, 1, 9, 2, 7],    [3, 1],    [1, 2, 3, 7, 8, 9],  "Ejemplo básico"),
        ([5, 4, 3, 2, 1],        [2, 1],    [1, 2, 3, 4, 5],     "Invertida gaps=[2,1]"),
        ([1],                    [1],        [1],                  "Un elemento"),
        ([],                     [1],        [],                   "Lista vacía"),
        ([3, 3, 3],              [1],        [3, 3, 3],            "Todos iguales"),
        (list(range(10,0,-1)),   [4, 2, 1], list(range(1,11)),    "n=10 con gaps=[4,2,1]"),
        ([2, 1, 4, 3, 6, 5],    [1],        [1, 2, 3, 4, 5, 6],  "gaps=[1] = Insertion Sort"),
    ]
    aprobados = 0
    for lista, gaps, esperado, desc in casos:
        t0 = time.perf_counter()
        try:
            resultado = fn(lista[:], gaps[:])
            t1 = time.perf_counter()
            if resultado == esperado:
                print(f"  ✅ {desc} ({(t1-t0)*1000:.2f}ms)")
                aprobados += 1
            else:
                print(f"  ❌ {desc} | Esperado: {esperado} | Obtenido: {resultado}")
        except Exception as e:
            print(f"  💥 {desc} — Error: {e}")
    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados == len(casos) else f'⚠️  {aprobados}/{len(casos)} casos correctos'}")

verificar_ejercicio_1(shell_sort_gaps)

In [ ]:
# ═══════════════════════════════════════════════════
# SOLUCIÓN — Descomenta para ver después de intentarlo
# ═══════════════════════════════════════════════════

# def shell_sort_gaps(lista: list, gaps: list) -> list:
#     """Shell Sort con secuencia de gaps explícita."""
#     a = lista[:]
#     n = len(a)
#     for gap in gaps:
#         for i in range(gap, n):
#             clave = a[i]
#             j = i - gap
#             while j >= 0 and a[j] > clave:
#                 a[j + gap] = a[j]
#                 j -= gap
#             a[j + gap] = clave
#     return a
# 
# # Nota: con gaps=[1], es exactamente Insertion Sort.
# # La magia está en que gaps grandes reducen el número de inversiones
# # para que el Insertion Sort final (gap=1) sea rápido.

## 🧪 Ejercicio 2 ⭐⭐: Counting Sort con Strings por Longitud

**Descripción:** Implementa `counting_sort_strings(lista)` que ordena una lista  
de strings por **longitud** (de menor a mayor). Debe ser **estable**: strings con  
la misma longitud mantienen su orden relativo original.

**Ejemplo:**
```
Entrada: ["banana", "pera", "kiwi", "manzana", "uva"]
Salida:  ["uva", "pera", "kiwi", "banana", "manzana"]
```

**Restricciones:** Las longitudes están en [0, 20]  
**Complejidad esperada:** O(n + k) donde k = longitud máxima

In [ ]:
def counting_sort_strings(lista: list) -> list:
    """
    Ordena lista de strings por longitud usando Counting Sort estable.

    Parámetros:
        lista (list): lista de strings
    Retorna:
        list: lista de strings ordenada por longitud (estable)
    """
    # Tu código aquí
    pass

In [ ]:
def verificar_ejercicio_2(fn):
    import time
    casos = [
        (["banana", "pera", "kiwi", "manzana", "uva"],
         ["uva", "pera", "kiwi", "banana", "manzana"],
         "Frutas por longitud"),
        ([], [], "Lista vacía"),
        (["a"], ["a"], "Un elemento"),
        (["abc", "def", "ghi"], ["abc", "def", "ghi"], "Misma longitud — estabilidad"),
        (["", "a", "bb", "ccc"], ["", "a", "bb", "ccc"], "Con string vacío"),
        (["python", "java", "c", "go", "rust"],
         ["c", "go", "java", "rust", "python"],
         "Lenguajes por longitud"),
    ]
    aprobados = 0
    for lista, esperado, desc in casos:
        t0 = time.perf_counter()
        try:
            resultado = fn(lista[:])
            t1 = time.perf_counter()
            if resultado == esperado:
                print(f"  ✅ {desc} ({(t1-t0)*1000:.2f}ms)")
                aprobados += 1
            else:
                print(f"  ❌ {desc}")
                print(f"     Esperado: {esperado}")
                print(f"     Obtenido: {resultado}")
        except Exception as e:
            print(f"  💥 {desc} — Error: {e}")
    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados == len(casos) else f'⚠️  {aprobados}/{len(casos)} casos correctos'}")

verificar_ejercicio_2(counting_sort_strings)

In [ ]:
# ═══════════════════════════════════════════════════
# SOLUCIÓN — Descomenta para ver después de intentarlo
# ═══════════════════════════════════════════════════

# def counting_sort_strings(lista: list) -> list:
#     """Counting Sort estable de strings por longitud."""
#     if not lista:
#         return []
#     k = max(len(s) for s in lista) + 1
#
#     # Paso 1: Contar frecuencias de cada longitud
#     conteo = [0] * k
#     for s in lista:
#         conteo[len(s)] += 1
#
#     # Paso 2: Calcular posiciones de inicio (prefix sum)
#     # conteo[i] = primera posición en la salida de strings de longitud i
#     for i in range(1, k):
#         conteo[i] += conteo[i - 1]
#
#     # Paso 3: Colocar en la salida de derecha a izquierda (garantiza estabilidad)
#     salida = [''] * len(lista)
#     for s in reversed(lista):
#         conteo[len(s)] -= 1
#         salida[conteo[len(s)]] = s
#
#     return salida
# 
# # Complejidad: O(n + k) temporal, O(n + k) espacial

## 🔬 Zona de Experimentación

Sugerencias:
- ¿Qué pasa con Counting Sort si k >> n? (p.ej. números de 0 a 1.000.000)
- ¿Cuál secuencia de gaps da menos comparaciones para datos casi ordenados?
- ¿Puedes implementar Shell Sort recursivo?

In [ ]:
# Espacio libre para experimentar

In [ ]:
# Espacio libre para experimentar

# Sección 7: Autoevaluación

In [ ]:
import ipywidgets as widgets
from IPython.display import display

preguntas = [
    {
        "pregunta": "¿Por qué Shell Sort con gaps grandes es más rápido que Insertion Sort?",
        "opciones": [
            "Porque usa menos memoria",
            "Porque reduce el número de inversiones antes del paso final gap=1",
            "Porque compara menos pares de elementos en total",
            "Porque es un algoritmo recursivo"
        ],
        "correcta": 1,
        "explicacion": "Los gaps grandes permiten mover elementos lejos de su posición en un solo paso, reduciendo inversiones. Cuando llega el gap=1 (Insertion Sort), la lista ya está casi ordenada y es rápida."
    },
    {
        "pregunta": "¿Cuál es la restricción fundamental de Counting Sort?",
        "opciones": [
            "Solo funciona con listas de tamaño n par",
            "Requiere que los elementos sean enteros en un rango [0,k) conocido",
            "Solo funciona con listas ya parcialmente ordenadas",
            "Necesita O(n log n) de memoria"
        ],
        "correcta": 1,
        "explicacion": "Counting Sort necesita 'indexar' por valor, lo que requiere que los valores sean enteros en un rango conocido. No puede usarse con datos de comparación arbitraria (strings, flotantes generales)."
    },
    {
        "pregunta": "La cota inferior Ω(n log n) aplica a:",
        "opciones": [
            "Todos los algoritmos de ordenamiento sin excepción",
            "Solo a algoritmos basados en comparaciones entre elementos",
            "Solo a algoritmos in-place (sin memoria extra)",
            "Solo a algoritmos recursivos"
        ],
        "correcta": 1,
        "explicacion": "La cota inferior del árbol de decisión solo aplica a algoritmos que deciden el orden comparando pares de elementos. Counting Sort la evita porque no compara — usa los valores directamente como índices."
    },
]

def crear_quiz(preguntas):
    for i, p in enumerate(preguntas):
        radio  = widgets.RadioButtons(options=p["opciones"], description=f"P{i+1}:",
                                      style={'description_width': 'initial'},
                                      layout={'width': 'max-content'})
        boton  = widgets.Button(description="Verificar", button_style="info")
        salida = widgets.Output()
        label  = widgets.HTML(f"<b>P{i+1}: {p['pregunta']}</b>")

        def verificar(b, r=radio, o=salida, c=p["correcta"], e=p["explicacion"], opts=p["opciones"]):
            with o:
                o.clear_output()
                if r.value == opts[c]:
                    print(f"✅ ¡Correcto! {e}")
                else:
                    print(f"❌ Respuesta: {opts[c]}\n   {e}")

        boton.on_click(verificar)
        display(widgets.VBox([label, radio, boton, salida]))
        print("─" * 70)

crear_quiz(preguntas)

# Sección 8: Lecturas y Recursos de Práctica

## Textbooks

| Libro | Edición | Capítulo | Tema |
|-------|---------|----------|------|
| Sedgewick & Wayne — *Algorithms* | 4ª ed. | Cap. 2.1 | Shell Sort |
| Sedgewick & Wayne — *Algorithms* | 4ª ed. | Cap. 2.7 | Bounds on sorting |
| Cormen et al. (CLRS) — *Intro to Algorithms* | 4ª ed. | Cap. 8.1–8.2 | Lower bounds + Counting Sort |
| Skiena — *The Algorithm Design Manual* | 3ª ed. | Cap. 4.2 | Bucketing, Counting, Radix |

## Recursos gratuitos
- 🎬 [Shell Sort — VisuAlgo](https://visualgo.net/en/sorting) — selecciona "Shell" en el menú
- 📄 [Paper original de Shell (1959)](https://dl.acm.org/doi/10.1145/368370.368387) — solo 2 páginas, vale la pena
- 🎬 [Lower bound for comparison sorting](https://www.youtube.com/watch?v=Nz1KZXbghj8) — MIT OpenCourseWare

## Práctica en Codeforces

| # | Problema | Rating | Por qué es útil |
|---|----------|--------|-----------------|
| 1 | [Elections](https://codeforces.com/problemset/problem/370/C) | ⭐ 900 | Counting Sort aplicado a votos |
| 2 | [Vasya and Socks](https://codeforces.com/problemset/problem/460/B) | ⭐ 900 | Ordenar y procesar por frecuencia |
| 3 | [k-Nearest](https://codeforces.com/problemset/problem/545/D) | ⭐⭐ 1200 | Ordenar + búsqueda por rango |
| 4 | [Pancake Sorting](https://codeforces.com/problemset/problem/926/D) | ⭐⭐⭐ 1500 | Variante de ordenamiento con operaciones restringidas |

⚠️ Los problemas 1 y 2 son el **mínimo esperado**. El 3 es intermedio. El 4 es desafío opcional.